In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "/data/anantaraha/amp/dataset/laion_art/output/cogvlm/exp3/exp3_image_scores.csv"

df = pd.read_csv(CSV_PATH)

metrics = ["A_MD", "A_DD", "S_gap"]
condition_order = ["source", "adversarial", "target"]

print(f"Rows: {len(df)}")
print(df["condition"].value_counts())

In [ ]:
summary = (
    df.groupby("condition")[metrics]
      .agg(["mean", "median", "std"])
      .reindex(condition_order)
)

# Make it compact
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.round(4)

display(summary)

In [ ]:
mean_table = (
    df.groupby("condition")[metrics]
      .mean()
      .reindex(condition_order)
      .round(4)
)

display(mean_table)

In [ ]:
paired = (
    df.pivot(index="sample_id", columns="condition", values="S_gap")
      .dropna(subset=["source", "adversarial"])
)

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(paired["source"], paired["adversarial"], s=45, alpha=0.8)

low = min(paired["source"].min(), paired["adversarial"].min())
high = max(paired["source"].max(), paired["adversarial"].max())

ax.plot([low, high], [low, high], "--", linewidth=1.5)

ax.set_xlabel(r"Source $S_{gap}$")
ax.set_ylabel(r"Adversarial $S_{gap}$")
ax.set_title(r"CogVLM: Per-image change in $S_{gap}$")

n_above = (paired["adversarial"] > paired["source"]).sum()
ax.text(
    0.05, 0.95,
    f"{n_above}/{len(paired)} adversarial samples above source",
    transform=ax.transAxes,
    va="top"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

all_scores = df["S_gap"].dropna()
bins = np.linspace(all_scores.min(), all_scores.max(), 20)

for condition in condition_order:
    values = df.loc[df["condition"] == condition, "S_gap"].dropna()
    ax.hist(
        values,
        bins=bins,
        alpha=0.5,
        label=f"{condition.capitalize()} (n={len(values)})"
    )

ax.set_xlabel(r"$S_{gap} = A_{DD} - A_{MD}$")
ax.set_ylabel("Number of images")
ax.set_title(r"CogVLM: Distribution of single-image $S_{gap}$")
ax.legend()

plt.tight_layout()
plt.show()